In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import plotly.express as px
import plotly.graph_objects as go

import sys
sys.path.append('../')
import plotting

In [ ]:
# define colors
linecolors = {
    '3mer': '#6baed6',
    '4mer': '#de2d26',
    '5mer': '#31a354',
    'SMOTE': '#de2d26',
    'regression_plus_probs': '#31a354',
    '1D-CNN': '#969696'
}
areacolors = {
    '3mer': 'rgba(107,174,214,0.2)', 
    '4mer': 'rgba(222,45,38,0.2)', 
    '5mer': 'rgba(49,163,84,0.2)', 
    'SMOTE': 'rgba(222,45,38,0.2)',
    'regression_plus_probs': 'rgba(49,163,84,0.2)',
    '1D-CNN': 'rgba(150,150,150,0.2)'
}

# define positions
positions_roc = {
    '3mer': (0.8, 0.6),
    '4mer': (0.8, 0.5),
    '5mer': (0.8, 0.4),
    'SMOTE': (0.8, 0.2),
    'regression_plus_probs': (0.8, 0.3),
    '1D-CNN': (0.8, 0.1),
}
positions_pr = {
    '3mer': (0.8, 0.9),
    '4mer': (0.8, 0.8),
    '5mer': (0.8, 0.7),
    'SMOTE': (0.8, 0.5),
    'regression_plus_probs': (0.8, 0.6),
    '1D-CNN': (0.8, 0.4),
}

# define plots
plots_by_methods = [
    ('kmer', ('3mer', '4mer', '5mer', '1D-CNN')),
    ('sampling', ('SMOTE', '1D-CNN')),
    ('regression', ('regression_plus_probs', '1D-CNN')),
]

# Internal - GCall and GCfix

In [ ]:
results = []
for dataset in ('GCall', 'GCfix'):
    for method in ('3mer', '4mer', '5mer', 'SMOTE', 'regression_plus_probs'):
        df = pd.read_csv(f'../data/machine_learning_results/additional_results/cleaned_Internal_{dataset}_2perc_{method}.csv')
        df['method'] = method
        df['dataset'] = dataset
        results.append(df)

# add 1D-CNN results
prediction_per_data, label_per_data = pd.read_pickle('../data/machine_learning_results/GCdata_internal_validation.pkl')
for dataset in ('GCall', 'GCfix'):
    for i in range(len(prediction_per_data[dataset])):
        df = pd.DataFrame.from_dict({
            'pred_probability': prediction_per_data[dataset][i],
            'binary_label': label_per_data[dataset][i],
            'fold': i+1,
            'method': '1D-CNN',
            'dataset': dataset
        })
        results.append(df)

df = pd.concat(results, ignore_index=True)

df

### ROC/PRC

In [ ]:
for plot_name, plot_methods in plots_by_methods:
    # create a set of figures for each dataset
    for dataset in df['dataset'].unique():
        # create empty figures for ROC and PR curves
        fig_roc = go.Figure()
        fig_pr = go.Figure()

        # empty dicts to store the metrics
        data_roc = {}
        data_pr = {}

        # go through all methods
        for method in plot_methods:
            # get the data for the current method
            df_method = df[(df['method'] == method) & (df['dataset'] == dataset)]
            
            tprs = []
            aucs = []
            precisions = []
            average_precisions = []
            base_recall = np.linspace(0, 1, 100)
            
            # calculate metrics per fold
            for i in df_method['fold'].unique():
                # get the predictions and labels for the current fold
                fold_predictions = df_method[df_method['fold'] == i]['pred_probability'].values
                fold_labels = df_method[df_method['fold'] == i]['binary_label'].values
                # calculate ROC curve and AUC
                fpr, tpr, _ = roc_curve(fold_labels, fold_predictions)
                roc_auc = auc(fpr, tpr)
                tprs.append(np.interp(base_recall, fpr, tpr))
                tprs[-1][0] = 0.0
                aucs.append(roc_auc)
                
                precision, recall, _ = precision_recall_curve(fold_labels, fold_predictions)
                precisions.append(np.interp(base_recall, recall[::-1], precision[::-1]))
                average_precisions.append(average_precision_score(fold_labels, fold_predictions))
            
            # Calculate mean and standard deviation for the metrics
            mean_tpr = np.mean(tprs, axis=0)
            std_tpr = np.std(tprs, axis=0)
            mean_auc = np.mean(aucs)
            mean_precision = np.mean(precisions, axis=0)
            std_precision = np.std(precisions, axis=0)
            mean_average_precision = np.mean(average_precisions)
            
            # save metrics
            data_roc[method] = {'fpr': base_recall, 'mean_tpr': mean_tpr, 'std_tpr': std_tpr}
            data_pr[method] = {'recall': base_recall, 'mean_precision': mean_precision, 'std_precision': std_precision}
            
            # ROC curve with uncertainty band   
            fig_roc.add_traces([
                go.Scatter(
                    x=base_recall, 
                    y=mean_tpr - std_tpr, 
                    line=dict(color='rgba(0,0,0,0)'),
                ),
                go.Scatter(
                    x=base_recall, 
                    y=mean_tpr + std_tpr,
                    line=dict(color='rgba(0,0,0,0)'),
                    fill='tonexty', 
                    fillcolor=areacolors[method],
                ),
                go.Scatter(
                    x=base_recall, 
                    y=mean_tpr, 
                    line=dict(color=linecolors[method]),
                ),
            ])
            fig_roc.add_annotation(
                x=positions_roc[method][0], 
                y=positions_roc[method][1], 
                text=f'{method}: {mean_auc:.2f}', 
                showarrow=False, 
                font_color=linecolors[method]
            )

            # Precision-recall curve with uncertainty band
            fig_pr.add_traces([
                go.Scatter(
                    x=base_recall, 
                    y=mean_precision - std_precision, 
                    line=dict(color='rgba(0,0,0,0)'),
                ),
                go.Scatter(
                    x=base_recall, 
                    y=mean_precision + std_precision,
                    line=dict(color='rgba(0,0,0,0)'),
                    fill='tonexty', 
                    fillcolor=areacolors[method],
                ),
                go.Scatter(
                    x=base_recall, 
                    y=mean_precision, 
                    line=dict(color=linecolors[method]),
                ),
            ])
            fig_pr.add_annotation(
                x=positions_pr[method][0], 
                y=positions_pr[method][1], 
                text=f'{method}: {mean_average_precision:.2f}', 
                showarrow=False, 
                font_color=linecolors[method]
            )


        fig_roc.update_layout(
            xaxis_title='False positive rate',
            yaxis_title='True positive rate',
            showlegend=False,
            margin=dict(l=0, r=5, t=5, b=0),
            width=160,
            height=160
        )
        fig_roc.update_yaxes(range=[0, 1.01])   
        fig_roc.update_xaxes(range=[0, 1])  
        fig_roc = plotting.standardize_plot(fig_roc)
        fig_roc.write_image(f"./SI_figure_additional_models/roc_curve_internal_{plot_name}_{dataset}.svg")
        fig_roc.show()


        fig_pr.update_layout(
            xaxis_title='Recall',
            yaxis_title='Precision',
            showlegend=False,
            margin=dict(l=0, r=5, t=5, b=0),
            width=160,
            height=160
        )
        fig_pr.update_yaxes(range=[0, 1.01])   
        fig_pr.update_xaxes(range=[0, 1])  
        fig_pr = plotting.standardize_plot(fig_pr)
        fig_pr.write_image(f"./SI_figure_additional_models/pr_curve_internal_{plot_name}_{dataset}.svg")
        fig_pr.show()

### Regression

In [ ]:
# create a set of figures for each dataset
for dataset in df['dataset'].unique():
    # create empty figures for ROC and PR curves
    fig = go.Figure()

    # get the data for the regression method
    df_method = df[(df['method'] == 'regression_plus_probs') & (df['dataset'] == dataset)]
    predictions = df_method['pred_efficiency'].values
    groundtruth = df_method['true_efficiency'].values
        
    # add the curve to the figure
    fig.add_traces([
        go.Scatter(
            y=predictions, 
            x=groundtruth, 
            line=dict(color='black'),
            marker=dict(size=3),
            mode='markers',
        ),
    ])

    # add the metrics to the figure
    r2 = np.corrcoef(predictions, groundtruth)[0, 1]**2
    fig.add_annotation(
        x=0.95, 
        y=0.866, 
        text=f'R<sup>2</sup>={r2:.3f}', 
        showarrow=False, 
        font_color='black'
    )
    rsme = np.sqrt(np.mean((predictions - groundtruth)**2))
    fig.add_annotation(
        x=0.95, 
        y=0.833, 
        text=f'RMSE={rsme:.3f}', 
        showarrow=False, 
        font_color='black'
    )
    rank_corr = np.corrcoef(predictions.argsort(), groundtruth.argsort())[0, 1]
    fig.add_annotation(
        x=0.95, 
        y=0.8, 
        text=f'Rank corr.={rank_corr:.3f}', 
        showarrow=False, 
        font_color='black'
    )

    # add a dashed x=y curve
    fig.add_traces([
        go.Scatter(
            x=[0.7, 1.1], 
            y=[0.7, 1.1], 
            line=dict(color='black', dash='dash', width=1),
        ),
    ])


    fig.update_layout(
        yaxis_title='Predicted efficiency',
        xaxis_title='Experimental efficiency',
        showlegend=False,
        margin=dict(l=0, r=5, t=5, b=0),
        width=160,
        height=160
    )
    fig.update_yaxes(range=[0.75, 1.05])   
    fig.update_xaxes(range=[0.75, 1.05])  
    fig = plotting.standardize_plot(fig)
    fig.write_image(f"./SI_figure_additional_models/xy_curve_internal_{dataset}.svg")
    fig.show()

# External - GCall2GCfix and GCfix2GCall

In [ ]:
results = []
for dataset in ('GCall2GCfix', 'GCfix2GCall'):
    for method in ('3mer', '4mer', '5mer', 'SMOTE', 'regression_plus_probs'):
        df = pd.read_csv(f'../data/machine_learning_results/additional_results/cleaned_External_{dataset}_2perc_{method}.csv')
        df['method'] = method
        df['dataset'] = dataset
        results.append(df)

# add 1D-CNN results
prediction_per_data, label_per_data = pd.read_pickle('../data/machine_learning_results/GCdata_external_validation.pkl')
for dataset in ('GCall -> GCfix', 'GCfix -> GCall'):
    df = pd.DataFrame.from_dict({
        'pred_probability': prediction_per_data[dataset],
        'binary_label': label_per_data[dataset],
        'method': '1D-CNN',
        'dataset': dataset.replace(' -> ', '2')
    })
    results.append(df)


df = pd.concat(results, ignore_index=True)

df

### ROC/PRC

In [ ]:
for plot_name, plot_methods in plots_by_methods:

    # create a set of figures for each dataset
    for dataset in df['dataset'].unique():
        # create empty figures for ROC and PR curves
        fig_roc = go.Figure()
        fig_pr = go.Figure()

        # empty dicts to store the metrics
        data_roc = {}
        data_pr = {}

        # go trough all datasets 
        for method in plot_methods:
            # get the data for the current method
            df_method = df[(df['method'] == method) & (df['dataset'] == dataset)]
            flat_predictions = df_method['pred_probability'].values
            flat_labels = df_method['binary_label'].values

            # compute metrics
            fpr, tpr, _ = roc_curve(flat_labels, flat_predictions)
            roc_auc = auc(fpr, tpr)
            precision, recall, _ = precision_recall_curve(flat_labels, flat_predictions)
            average_precision = average_precision_score(flat_labels, flat_predictions)

            # save metrics
            data_roc[method] = {'fpr': fpr, 'tpr': tpr}
            data_pr[method] = {'precision': precision, 'recall': recall}

            # ROC curve 
            fig_roc.add_traces([
                go.Scatter(
                    x=fpr, 
                    y=tpr, 
                    line=dict(color=linecolors[method]),
                ),
            ])
            fig_roc.add_annotation(
                x=positions_roc[method][0], 
                y=positions_roc[method][1], 
                text=f'{method}: {roc_auc:.2f}', 
                showarrow=False, 
                font_color=linecolors[method]
            )

            # Precision-recall curve
            fig_pr.add_traces([
                go.Scatter(
                    x=recall, 
                    y=precision, 
                    line=dict(color=linecolors[method]),
                ),
            ])
            fig_pr.add_annotation(
                x=positions_pr[method][0], 
                y=positions_pr[method][1], 
                text=f'{method}: {average_precision:.2f}', 
                showarrow=False, 
                font_color=linecolors[method]
            )

        fig_roc.update_layout(
            xaxis_title='False positive rate',
            yaxis_title='True positive rate',
            showlegend=False,
            margin=dict(l=0, r=5, t=5, b=0),
            width=160,
            height=160
        )
        fig_roc.update_yaxes(range=[0, 1.01])   
        fig_roc.update_xaxes(range=[0, 1])  
        fig_roc = plotting.standardize_plot(fig_roc)
        fig_roc.write_image(f"./SI_figure_additional_models/roc_curve_external_{plot_name}_{dataset}.svg")
        fig_roc.show()


        fig_pr.update_layout(
            xaxis_title='Recall',
            yaxis_title='Precision',
            showlegend=False,
            margin=dict(l=0, r=5, t=5, b=0),
            width=160,
            height=160
        )
        fig_pr.update_yaxes(range=[0, 1.01])
        fig_pr.update_xaxes(range=[0, 1])
        fig_pr = plotting.standardize_plot(fig_pr)
        fig_pr.write_image(f"./SI_figure_additional_models/pr_curve_external_{plot_name}_{dataset}.svg")
        fig_pr.show()


### Regression

In [ ]:
# create a set of figures for each dataset
for dataset in df['dataset'].unique():
    # create empty figures for ROC and PR curves
    fig = go.Figure()

    # get the data for the regression method
    df_method = df[(df['method'] == 'regression_plus_probs') & (df['dataset'] == dataset)]
    predictions = df_method['pred_efficiency'].values
    groundtruth = df_method['true_efficiency'].values
        
    # add the curve to the figure
    fig.add_traces([
        go.Scatter(
            y=predictions, 
            x=groundtruth, 
            line=dict(color='black'),
            marker=dict(size=3),
            mode='markers',
        ),
    ])

    # add the metrics to the figure
    r2 = np.corrcoef(predictions, groundtruth)[0, 1]**2
    fig.add_annotation(
        x=0.99, 
        y=0.866, 
        text=f'R<sup>2</sup>={r2:.3f}', 
        showarrow=False, 
        font_color='black'
    )
    rsme = np.sqrt(np.mean((predictions - groundtruth)**2))
    fig.add_annotation(
        x=0.99, 
        y=0.833, 
        text=f'RMSE={rsme:.3f}', 
        showarrow=False, 
        font_color='black'
    )
    rank_corr = np.corrcoef(predictions.argsort(), groundtruth.argsort())[0, 1]
    fig.add_annotation(
        x=0.99, 
        y=0.8, 
        text=f'Rank corr.={rank_corr:.3f}', 
        showarrow=False, 
        font_color='black'
    )

    # add a dashed x=y curve
    fig.add_traces([
        go.Scatter(
            x=[0.7, 1.2], 
            y=[0.7, 1.2], 
            line=dict(color='black', dash='dash', width=1),
        ),
    ])


    fig.update_layout(
        yaxis_title='Predicted efficiency',
        xaxis_title='Experimental efficiency',
        showlegend=False,
        margin=dict(l=0, r=5, t=5, b=0),
        width=160,
        height=160
    )
    fig.update_yaxes(range=[0.75, 1.1])   
    fig.update_xaxes(range=[0.75, 1.1])  
    fig = plotting.standardize_plot(fig)
    fig.write_image(f"./SI_figure_additional_models/xy_curve_external_{dataset}.svg")
    fig.show()